# CO5085 A1 · ResNet18 and Swin-T on CIFAR-100

How do pretraining and the extent of fine-tuning affect image classification accuracy and computational cost? This study compares ResNet18 and Swin-T on CIFAR-100, followed by a controlled comparison of Swin-T initializations.

## 1. Libraries and configuration

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p / "src/A1/protocol.py").is_file()), None)
if root is None:
    if "google.colab" not in sys.modules:
        raise FileNotFoundError(
            "Open the notebook from the CO5085-deep-learning repository."
        )
    root = Path("/content/CO5085-deep-learning")
    if not (root / "src/A1/protocol.py").is_file():
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/dathuynh1108/CO5085-deep-learning.git",
                str(root),
            ],
            check=True,
        )
sys.path.insert(0, str(root / "src"))
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, display
from torchvision.datasets import CIFAR100
from torchvision.datasets.utils import download_url
from A1.data import development_data, loader
from A1.engine import epoch_pass, test_run, train_run
from A1.models import build_model
from A1.protocol import load_protocol
from common.runtime import atomic_json, choose_device


### Experimental settings

The main experiment compares eight configurations over 20 epochs: random initialization and three pretrained fine-tuning regimes for each architecture. The Swin-T follow-up compares random and ImageNet initialization under the same 40-epoch schedule. Both experiments use seeds 42, 43, and 44, the same split, and the same preprocessing.

In [ ]:
settings = {
    "assignment1": root / "configs/a1_experiments.json",
    "a1_swin_followup": root / "configs/a1_swin_followup.json",
}
protocol_names = os.getenv("CO5085_A1_PROTOCOLS", ",".join(settings)).split(",")
seeds = [int(seed) for seed in os.getenv("CO5085_A1_SEEDS", "42,43,44").split(",")]
output_root = Path(
    os.getenv("CO5085_A1_OUTPUT_ROOT", str(root / "runs/notebooks"))
).resolve()
data_root = Path(os.getenv("CO5085_A1_DATA_ROOT", str(root / "data"))).resolve()
workers = int(os.getenv("CO5085_A1_WORKERS", "8"))
resume = os.getenv("CO5085_A1_RESUME", "0") == "1"
confirm_test = os.getenv("CO5085_A1_CONFIRM_TEST", "0") == "1"


In [ ]:
device = choose_device(os.getenv("CO5085_A1_DEVICE", "cuda"))
if device.type == "cuda" and not torch.cuda.is_bf16_supported(
    including_emulation=False
):
    raise RuntimeError("The A1 protocol requires a GPU with native BF16 support.")
if len(set(seeds)) != len(seeds) or any(seed < 0 for seed in seeds) or workers < 0:
    raise ValueError(
        "Seeds must be distinct and nonnegative, and the worker count must be nonnegative."
    )
protocols = {name: load_protocol(settings[name]) for name in protocol_names}
output_root.mkdir(parents=True, exist_ok=True)
print(f"Device: {device} · seeds: {seeds} · output directory: {output_root}")
for name, protocol in protocols.items():
    print(
        name,
        protocol["epochs"],
        "epochs",
        len(protocol["experiments"]),
        "configurations",
    )


## 2. Data exploration and preprocessing

CIFAR-100 contains RGB images of size 32 × 32 from 100 classes. The validation split contains 50 images per class from the official training set. Training images are randomly cropped and flipped before resizing to 96 × 96. Validation and test images use deterministic transformations. The following plots show class counts and examples of the original images.

In [ ]:
if not (data_root / CIFAR100.base_folder / "meta").is_file():
    # This mirror preserves the original Toronto archive checksum.
    mirror = (
        "https://zenodo.org/records/10089977/files/cifar-100-python.tar.gz?download=1"
    )
    download_url(
        mirror, str(data_root), filename=CIFAR100.filename, md5=CIFAR100.tgz_md5
    )
    CIFAR100(str(data_root), train=True, download=True)

first = protocols[protocol_names[0]]
train_data, val_data, split, data_identity = development_data(data_root, first)
pixels = train_data.dataset.data
labels = np.asarray(train_data.dataset.targets)
train_counts = np.bincount(labels[split["train"]], minlength=100)
val_counts = np.bincount(labels[split["val"]], minlength=100)
assert len(train_data) == 45_000 and len(val_data) == 5_000
print(f"{len(train_data):,} training images and {len(val_data):,} validation images")
print(
    "Training-set RGB mean:", np.round(pixels[split["train"]].mean((0, 1, 2)) / 255, 4)
)


### Class balance

Class counts are shown for the training and validation splits.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(np.arange(100), train_counts, label="Training")
ax.bar(np.arange(100), val_counts, label="Validation")
ax.set(
    xlabel="Class index",
    ylabel="Number of images",
    title="CIFAR-100 class distribution",
)
ax.legend()
fig.tight_layout()
fig.savefig(output_root / "a1-class-counts.png", dpi=130)
display(Image(filename=str(output_root / "a1-class-counts.png")))
plt.close(fig)


### Original images

Eight training images illustrate the visual detail available at the original resolution.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
examples = np.random.default_rng(seeds[0]).choice(
    split["train"], len(axes.flat), replace=False
)
for ax, index in zip(axes.flat, examples):
    ax.imshow(pixels[index])
    ax.set_title(split["classes"][labels[index]], fontsize=9)
    ax.axis("off")
fig.tight_layout()
fig.savefig(output_root / "a1-samples.png", dpi=130)
display(Image(filename=str(output_root / "a1-samples.png")))
plt.close(fig)


## 3. Training and checkpoint selection

Each model has a classification head with 100 outputs. Training uses cross-entropy, gradient clipping, and AdamW. Pretrained models update only the head, the final stage and head, or all parameters. Validation accuracy selects the checkpoint, with cross-entropy used to break ties.

In [ ]:
jobs = []
for name, protocol in protocols.items():
    for experiment in protocol["experiments"]:
        for seed in seeds:
            run = output_root / name / experiment["id"] / f"seed_{seed}"
            options = dict(
                run=run,
                protocol=protocol,
                experiment=experiment,
                seed=seed,
                data_root=data_root,
                device=device,
                workers=workers,
                config=settings[name],
            )
            print(f"{name} · {experiment['id']} · seed {seed}", flush=True)
            metadata = train_run(**options, resume=resume)
            assert metadata["status"] == "trained"
            jobs.append((name, options))


## 4. Evaluation

Held-out test evaluation follows protocol finalization and validation-based checkpoint selection. Results identify the data split, training seed, and selected epoch, alongside accuracy, macro-F1, trainable parameters, training time, and peak GPU memory.

In [ ]:
rows, matrices = [], {}
for name, options in jobs:
    run = options["run"]
    preset, seed = options["experiment"], options["seed"]
    record = json.loads((run / "metadata.json").read_text())
    if confirm_test:
        metrics = test_run(**options)
    else:
        model = build_model(preset, initialize_pretrained=False).to(device)
        checkpoint = torch.load(run / "best.pt", map_location="cpu", weights_only=False)
        model.load_state_dict(checkpoint["model"])
        batches = loader(
            val_data,
            options["protocol"]["batch_size"],
            workers,
            torch.Generator().manual_seed(seed + 1),
            False,
            device,
        )
        metrics = epoch_pass(model, batches, device)
        del model, checkpoint, batches
    label = (name, preset["id"])
    matrices[label] = matrices.get(label, 0) + np.asarray(metrics["confusion"])
    rows.append(
        {
            "Protocol": name,
            "Configuration": preset["id"],
            "Seed": seed,
            "Split": "Test" if confirm_test else "Validation",
            "Selected epoch": record["best_validation"]["epoch"],
            "Accuracy": metrics["accuracy"],
            "Macro-F1": metrics["macro_f1"],
            "Trainable parameters": record["trainable_params"],
            "Training time (s)": record["train_seconds"],
            "VRAM MiB": record["peak_gpu_memory_bytes"] / 2**20,
        }
    )


### Metrics across seeds

Accuracy and macro-F1 are summarized by configuration using the mean and sample standard deviation.

In [ ]:
results = pd.DataFrame(rows)
display(results.round(4))
display(
    results.groupby(["Protocol", "Configuration"])[["Accuracy", "Macro-F1"]]
    .agg(["mean", "std"])
    .round(4)
)
results.to_csv(
    output_root / f'a1-metrics-seeds-{"-".join(map(str, seeds))}.csv', index=False
)
atomic_json(
    output_root / f'a1-config-seeds-{"-".join(map(str, seeds))}.json',
    {
        "seeds": seeds,
        "protocols": protocols,
        "data": data_identity,
        "test": confirm_test,
        "runs": [str(options["run"]) for _, options in jobs],
    },
)


## 5. Learning curves and confusion matrices

Learning curves average the evaluated seeds. Confusion matrices are normalized by the true class. The accompanying table identifies the most frequently confused class pair for each model.

In [ ]:
for name, protocol in protocols.items():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for preset in protocol["experiments"]:
        histories = []
        for seed in seeds:
            history = output_root / name / preset["id"] / f"seed_{seed}" / "history.csv"
            histories.append(pd.read_csv(history))
        for ax, metric, title in zip(
            axes,
            ["val_accuracy", "val_ce"],
            ["Validation accuracy", "Validation cross-entropy"],
        ):
            values = np.stack([history[metric] for history in histories])
            ax.plot(histories[0]["epoch"], values.mean(0), label=preset["id"])
            ax.set(xlabel="Epoch", ylabel=title)
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    path = output_root / f'{name}-curves-{"-".join(map(str, seeds))}.png'
    fig.savefig(path, dpi=150)
    display(Image(filename=str(path)))
    plt.close(fig)


### Confused classes

Rows denote true labels and columns denote predicted labels. The table reports the largest off-diagonal count for each model.

In [ ]:
for name, protocol in protocols.items():
    ids = [
        preset["id"]
        for preset in protocol["experiments"]
        if preset["regime"] in ("scratch", "full")
    ]
    fig, axes = plt.subplots(1, len(ids), figsize=(4 * len(ids), 4), squeeze=False)
    mistakes = []
    for ax, preset_id in zip(axes.flat, ids):
        matrix = matrices[name, preset_id]
        proportions = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
        ax.imshow(proportions, cmap="Blues", vmin=0, vmax=1)
        ax.set(title=preset_id, xlabel="Predicted label", ylabel="True label")
        off_diagonal = matrix.copy()
        np.fill_diagonal(off_diagonal, 0)
        actual, predicted = np.unravel_index(off_diagonal.argmax(), matrix.shape)
        mistakes.append(
            (
                preset_id,
                split["classes"][actual],
                split["classes"][predicted],
                int(off_diagonal[actual, predicted]),
            )
        )
    fig.tight_layout()
    path = output_root / f'{name}-confusion-{"-".join(map(str, seeds))}.png'
    fig.savefig(path, dpi=150)
    display(Image(filename=str(path)))
    plt.close(fig)
    display(
        pd.DataFrame(
            mistakes,
            columns=["Configuration", "True label", "Predicted label", "Error count"],
        )
    )


## 6. Interpretation and limitations

The 20-epoch experiment compares pretraining and fine-tuning regimes within a fixed training budget. The 40-epoch experiment holds the learning rate, optimizer, and training duration constant between Swin-T initializations. Comparisons across schedules must account for simultaneous changes to warmup, learning rate, and weight decay. Resizing the original 32 × 32 images adds no image detail. Identical images across splits are a further limitation examined in the report.